# Chapter 6: 構造化出力（Structured Output） 【Bedrock版】

エージェントの出力を Zod スキーマで型付きオブジェクトとして受け取り、プログラムで扱えるデータにします。

**使用モデル**: Amazon Nova Lite

## セットアップ

In [ ]:
import subprocess, os

def run(cmd):
    """UTF-8安全なコマンド実行（%%bashの代替）"""
    result = subprocess.run(cmd, shell=True, cwd=os.path.join(os.getcwd(), '..', '..'),
                            capture_output=True, text=True)
    if result.stdout: print(result.stdout, end='')
    if result.returncode != 0 and result.stderr: print(result.stderr, end='')

In [ ]:
run("echo Node.js: $(node -v) && npm install")

## 認証の確認

AWS Bedrock版はIAM Roleで認証します。SageMaker Studio上ではIAM Roleが自動で設定されています。

OpenAI APIキーは**不要**です。

In [ ]:
run('echo "AWS Region: ${AWS_REGION:-${AWS_DEFAULT_REGION:-us-east-1}}" && \
    aws sts get-caller-identity --query Arn --output text')

## ポイント解説

Zod スキーマで出力の型を定義し、型付きオブジェクトとして受け取ります。

```typescript
// ブログ記事の出力構造を定義
const BlogArticleSchema = z.object({
  title: z.string(),
  sections: z.array(z.object({
    heading: z.string(),
    body: z.string(),
  })),
  tags: z.array(z.string()),
});

const result = await agent.generate(messages, {
  structuredOutput: { schema: BlogArticleSchema },  // ← スキーマを渡す
});

// 型付きオブジェクトとしてアクセス
const article = BlogArticleSchema.parse(result.object);
console.log(article.title);              // string
console.log(article.sections[0].heading); // string
```

JSON シリアライズ可能なので、DB 保存や API 返却等の後続処理にそのまま使えます。

## Chapter 6の実行
**お題**: TypeScriptの型システムを活用したバグ防止テクニック


In [ ]:
run("npm run ch6:bedrock")

## 観察ポイント

1. 出力が自由テキストではなく、JSON オブジェクトになっているか？
2. title, sections, tags などのフィールドにプログラムでアクセスできるか？
3. 後続処理（DB保存、API返却等）に使える形式か？

## コード（参考）

In [ ]:
run("cat src/chapter6/run.ts")

## 次のステップ

→ `chapter7-mcp-server.ipynb`を開いてください。